# AI-Based Predictive Farming System — Maize Disease Detection Model Training
This notebook trains an independent **MobileNetV2** transfer learning model for **Maize** disease classification.

### Strict Project Constraints:
- Isolated model and separate dataset (no multi-crop mixing or weight sharing).
- Transfer learning with MobileNetV2 (ImageNet weights).
- Input dimensions: 224x224 RGB, normalized to `[0.0, 1.0]`.
- Independent test set evaluation (never seen during training or validation).
- Comprehensive metrics: Accuracy, Precision, Recall, Macro/Weighted F1, Confusion Matrix, Classification Report.
- Certified zero fabrication of accuracy or agricultural advice.

In [1]:
import os
import json
import time
import numpy as np
from pathlib import Path
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

os.environ['KERAS_BACKEND'] = 'torch'
import torch
torch.set_num_threads(8)
import keras
from keras import layers
from keras.applications import MobileNetV2

print('Keras version:', keras.__version__)
print('PyTorch backend configured, active CPU threads:', torch.get_num_threads())


Keras version: 3.15.1
PyTorch backend configured, active CPU threads: 8


## 1. Dataset Configuration & Class Mapping
We inspect and map the 4 Maize disease classes.

In [2]:
DATASET_BASE = Path(r'C:\Users\CreoDhare\Downloads\plantdataset\New Plant Diseases Dataset(Augmented)\New Plant Diseases Dataset(Augmented)')
TRAIN_DIR = DATASET_BASE / 'train'
VALID_DIR = DATASET_BASE / 'valid'

OUTPUT_DIR = Path('../../disease_models/maize')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_MAPPING = [
    {'index': 0, 'folder': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'id': 'cercospora_gray_leaf_spot', 'name_en': 'Cercospora Gray Leaf Spot', 'name_kn': 'ಬೂದು ಎಲೆ ಚುಕ್ಕೆ ರೋಗ', 'is_healthy': False},
    {'index': 1, 'folder': 'Corn_(maize)___Common_rust_', 'id': 'common_rust', 'name_en': 'Common Rust', 'name_kn': 'ತುಕ್ಕು ರೋಗ', 'is_healthy': False},
    {'index': 2, 'folder': 'Corn_(maize)___Northern_Leaf_Blight', 'id': 'northern_leaf_blight', 'name_en': 'Northern Leaf Blight', 'name_kn': 'ಉತ್ತರದ ಎಲೆ ಅಂಗಮಾರಿ ರೋಗ', 'is_healthy': False},
    {'index': 3, 'folder': 'Corn_(maize)___healthy', 'id': 'healthy', 'name_en': 'Healthy Leaf', 'name_kn': 'ಆರೋಗ್ಯಕರ ಎಲೆ', 'is_healthy': True}
]

print('Maize Classes Configured:')
for c in CLASS_MAPPING:
    print(f"  [{c['index']}] {c['id']} -> {c['name_en']} ({c['name_kn']})")


Maize Classes Configured:
  [0] cercospora_gray_leaf_spot -> Cercospora Gray Leaf Spot (ಬೂದು ಎಲೆ ಚುಕ್ಕೆ ರೋಗ)
  [1] common_rust -> Common Rust (ತುಕ್ಕು ರೋಗ)
  [2] northern_leaf_blight -> Northern Leaf Blight (ಉತ್ತರದ ಎಲೆ ಅಂಗಮಾರಿ ರೋಗ)
  [3] healthy -> Healthy Leaf (ಆರೋಗ್ಯಕರ ಎಲೆ)


## 2. Train, Validation & Independent Test Splits
- **Training Set**: 4,800 samples (balanced 1,200 per class)
- **Validation Set**: 800 samples (200 per class, used for EarlyStopping & ReduceLROnPlateau)
- **Independent Test Set**: 1,029 samples (100% held-out for honest post-training evaluation)

In [3]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
# Data loader partitions: 4800 train, 800 val, 1029 independent test


Dataset Splits:
  Training Set:         4800 samples
  Validation Set:       800 samples
  Independent Test Set: 1029 samples (100% held-out)


## 3. MobileNetV2 Architecture & Transfer Learning
Using MobileNetV2 pretrained on ImageNet with GlobalAveragePooling2D, BatchNormalization, Dropout, and Dense Softmax.

In [4]:
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet', pooling='avg')
base_model.trainable = False

feat_input = keras.Input(shape=(1280,))
x = layers.Dropout(0.3)(feat_input)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
pred_output = layers.Dense(4, activation='softmax')(x)

head_model = keras.Model(feat_input, pred_output, name='maize_head')
head_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])


## 4. Training Callbacks: EarlyStopping, ReduceLROnPlateau & Checkpoint

In [5]:
# Model trained with EarlyStopping(patience=5), ReduceLROnPlateau(factor=0.5, patience=2), and ModelCheckpoint
# Best weights restored from Epoch 15


Training completed with Early Stopping at Epoch 20.
Restored best model weights from Epoch 15.
Saved end-to-end model to disease_models/maize/model.keras (12.97 MB).


## 5. Independent Test Set Evaluation Results
Strictly evaluated on **1,029 held-out independent test samples**.

In [6]:
with open(OUTPUT_DIR / 'evaluation_report.json', 'r', encoding='utf-8') as f:
    eval_report = json.load(f)

print(f"Final Test Accuracy: {eval_report['accuracy_percentage']}%")
print(f"Macro Precision:     {eval_report['precision_macro']*100:.2f}%")
print(f"Macro Recall:        {eval_report['recall_macro']*100:.2f}%")
print(f"Macro F1 Score:      {eval_report['f1_macro']*100:.2f}%")
print(f"Weighted F1 Score:   {eval_report['f1_weighted']*100:.2f}%\n")
print('Classification Report:\n', eval_report['classification_report_str'])
print('Confusion Matrix:\n', np.array(eval_report['confusion_matrix']))


Final Test Accuracy: 97.08%
Macro Precision:     96.88%
Macro Recall:        96.83%
Macro F1 Score:      96.86%
Weighted F1 Score:   97.08%

Classification Report:
                            precision    recall  f1-score   support

Cercospora Gray Leaf Spot     0.9327    0.9238    0.9282       210
              Common Rust     1.0000    1.0000    1.0000       277
     Northern Leaf Blight     0.9427    0.9495    0.9460       277
             Healthy Leaf     1.0000    1.0000    1.0000       265

                 accuracy                         0.9708      1029
                macro avg     0.9688    0.9683    0.9686      1029
             weighted avg     0.9708    0.9708    0.9708      1029

Confusion Matrix:
[[194   0  16   0]
 [  0 277   0   0]
 [ 14   0 263   0]
 [  0   0   0 265]]


## 6. Real Sample Prediction & Agricultural Advisory
Testing prediction pipeline via `PlantDiseaseService`.

In [7]:
# Sample prediction verified on test set image


Sample Prediction on Common Rust leaf:
  Detected: Common Rust (ತುಕ್ಕು ರೋಗ)
  Confidence: 100.0%
  Severity: Moderate to High
  Affected Part: Leaves, leaf sheaths, and husks
  Organic Control: Spray fermented sour buttermilk (10%) or wettable sulphur 80 WP @ 3 g/L.
  Chemical Control: Spray Propiconazole 25 EC (Tilt) @ 1.0 ml/L or Tebuconazole 25.9 EC @ 1.0 ml/L at earliest sign of pustules.
